# 00.04 · Secrets and `.env` hygiene

**Module 00 · Foundations, Tooling & Research Skills** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 60–75 min |
| 📋 Prerequisites | Lessons 00.01–00.03, the `lc-mastery` environment, a free [Groq account](https://console.groq.com/). |
| 🔑 Keys | You create `GROQ_API_KEY` (free) in this lesson |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**.
> Run cells **from top to bottom**. If anything looks strange, use **Kernel → Restart Kernel and Run All Cells…**

## Learning objectives

1. **Create** a Groq API key and store it in a `.env` file that git ignores.
2. **Load** `.env` with `python-dotenv` and explain how the file is found and which value wins (`override`).
3. **Check** that keys exist **without ever printing them**, using `mastery.config`.
4. **Name** the environment variables that LangChain, LangSmith, and Groq read.
5. **Respond** correctly to a leaked key (rotate first, clean up second).

## 📚 Official reading (verified 2026-10-11)

| Page | Why |
|---|---|
| [python-dotenv](https://pypi.org/project/python-dotenv/) | `load_dotenv`, `dotenv_values`, `override`, the `%dotenv` magic |
| [The Twelve-Factor App: Config](https://12factor.net/config) | Why config lives in the environment, not in code |
| [ChatGroq setup](https://docs.langchain.com/oss/python/integrations/chat/groq) | `GROQ_API_KEY`, the `getpass` pattern |
| [Groq quickstart](https://console.groq.com/docs/quickstart) | Creating and exporting a Groq key |
| [LangSmith tracing quickstart](https://docs.langchain.com/langsmith/observability-quickstart) | `LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`, `LANGSMITH_ENDPOINT` |
| [Python `getpass`](https://docs.python.org/3/library/getpass.html) | Typing a secret without echoing it |
| Course: [`src/mastery/config.py`](../../src/mastery/config.py) · [`.env.example`](../../.env.example) | The helper you will use in every notebook |

---
## 1. Why secrets need a system

An **API key** is a password that spends money and acts as you. Anyone who has your Groq key can use your quota (and, on a paid plan, your credit card).

The twelve-factor rule: keep **config** (anything that changes between your laptop, CI, and production, such as keys and URLs) **out of the code**, in environment variables. Its litmus test: your code *"could be made open source at any moment, without compromising any credentials"* ([12factor.net/config](https://12factor.net/config)). This repository is public, so that test is literal for us.

| File | Committed? | Contains |
|---|---|---|
| `.env.example` | ✅ yes | Every variable name, with comments and **empty** values |
| `.env` | ❌ never (in `.gitignore`) | Your real values, on your machine only |

`python-dotenv` reads `.env` and copies its values into the process environment (`os.environ`), where libraries such as `langchain-groq` look for them.

---
## 2. Get a Groq API key (5 minutes, free)

1. Go to **https://console.groq.com/** and sign in (Google, GitHub, or email).
2. Open **API Keys**: **https://console.groq.com/keys**.
3. Click **Create API Key**. Name it after the machine, for example `lc-mastery-laptop`, so you know which key to revoke if that laptop is lost.
4. **Copy the key now** and paste it straight into `.env` (next step). Treat it as shown once: if you lose it, delete it and create a new one.

> Never paste a key into a notebook cell, a chat, a screenshot, or a commit message.

## 3. Create `.env` in Git Bash

In **Git Bash**, in the repo root (the folder with `.env.example`):

```bash
cp .env.example .env          # copy the template
ls -a | grep env              # you should see .env and .env.example
git check-ignore -v .env      # proves git ignores it
notepad .env                  # or: code .env   (VS Code)
```

**You should now see** `git check-ignore` print something like `.gitignore:9:.env	.env`: the file name, line number, and rule in `.gitignore` that ignores `.env`. **If it prints nothing, stop**: the file is *not* ignored. Check that you are in the repo root and that `.gitignore` is intact.

In the editor, find the line `GROQ_API_KEY=` and paste your key right after `=`:

```text
GROQ_API_KEY=gsk_your_real_key_goes_here
```

Course style: `NAME=value`, one variable per line, no quotes. (python-dotenv also accepts spaces around `=`, quotes, and `# comments`, but other tools that read `.env` files are stricter, so keep it simple.) Save the file. Leave the other keys empty for now.

> **Windows gotchas:** File Explorer hides extensions, so make sure the file is called `.env`, not `.env.txt` (`ls -a` tells the truth).
> **macOS/Linux:** same commands; open the file with `nano .env` or `code .env`.

---
## Setup

Make sure `python-dotenv` is present in the course environment (the create-course-env guide installed it with `mastery`; running this again is harmless). `%pip` installs into **this kernel's** environment.

In [ ]:
%pip install --quiet "python-dotenv>=1.2.4,<2"

In [ ]:
# Setup 1: Find the repository root.
# WHY: Jupyter starts this kernel in the notebook's own folder (modules/00-foundations/),
# but .env, data/ and src/ live at the repo root, two levels up.
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find the course repo (pyproject.toml). "
        "Start Jupyter from inside your clone of langchain-langgraph-mastery (or set LC_MASTERY_ROOT to it)."
    )

# Setup 2: Import the course package `mastery` from THIS clone's src/.
# WHY: pytest imports it from src/ too (pythonpath = ["src"] in pyproject.toml), so notebooks
# and tests run the same code even if this env was installed from another copy of the repo.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")  # The env's own install; terminal commands use it.
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root :", REPO_ROOT)
print("mastery   :", mastery.__version__, "from", mastery.__file__)

# Setup 3: Check that the env's own install (create-course-env guide, Step 5) points here too.
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder.")
    print("   Kernel → Restart Kernel, then run the notebook from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

**You should now see** the repo root and `mastery : 0.1.0 from …`, with a path that ends in `src\mastery\__init__.py` (`src/mastery/__init__.py` on macOS/Linux) **inside this repo**. The cell always imports `mastery` from this clone's `src/`, the same code `python -m pytest` tests. If you see a ⚠️ message instead, your environment's own install is missing or points to another copy of the repo: the lesson still works, but run the fix it prints once (create-course-env guide, Step 5), because terminal commands use that install.

---
## 4. How python-dotenv works (with fake values)

We first experiment in a **temporary folder with fake values**, so nothing here touches your real `.env`.

Two functions matter ([python-dotenv docs](https://pypi.org/project/python-dotenv/)):

| Function | Reads the file | Changes `os.environ`? | Use it for |
|---|---|---|---|
| `dotenv_values(path)` | ✅ | ❌ returns a dict | Inspecting or merging files |
| `load_dotenv(path)` | ✅ | ✅ adds missing variables | Normal use: libraries read `os.environ` |

In [ ]:
import os
import tempfile

from dotenv import dotenv_values, load_dotenv

# Step 1: Write a throwaway .env with FAKE values in a temporary folder.
demo_dir = Path(tempfile.mkdtemp(prefix="dotenv-demo-"))
demo_env = demo_dir / ".env"
demo_env.write_text(
    "DEMO_COLOR=blue\n"
    "DEMO_API_KEY=gsk_FAKE_demo_value_not_a_real_key_123456\n"
    "# Comments and blank lines are ignored\n"
    "DEMO_URL=https://${DEMO_COLOR}.example.com\n",   # ${VAR} interpolation
    encoding="utf-8",
)

# Step 2: dotenv_values() only READS. os.environ is untouched.
values = dotenv_values(demo_env)
print("Keys in the file   :", list(values))
print("DEMO_URL expanded  :", values["DEMO_URL"])
print("In os.environ yet? :", "DEMO_COLOR" in os.environ)

`${DEMO_COLOR}` was expanded inside `DEMO_URL`. python-dotenv expands `${VAR}` (with braces), not bare `$VAR`.

Now the important rule: **by default, the real environment wins** (`override=False`). If a variable is already set, for example because you exported it in Git Bash, the file does *not* replace it.

The same rule applies **inside** `${VAR}`: if `DEMO_COLOR=red` is already exported, `load_dotenv()` expands `DEMO_URL` to `https://red.example.com`, while `dotenv_values()` (which only reads the file) gives `https://blue.example.com`. `override=True` makes both use the file.

In [ ]:
# Step 3: Simulate a value that already exists in the environment (e.g. exported in Git Bash).
os.environ["DEMO_COLOR"] = "red"

load_dotenv(demo_env)                       # override=False is the default
print("After load_dotenv()              :", os.environ["DEMO_COLOR"])   # red: environment wins

load_dotenv(demo_env, override=True)        # explicit override
print("After load_dotenv(override=True) :", os.environ["DEMO_COLOR"])   # blue: file wins

# Step 4: Clean up so the demo leaves no trace in this kernel.
for name in ("DEMO_COLOR", "DEMO_API_KEY", "DEMO_URL"):
    os.environ.pop(name, None)
print("Demo variables removed:", not any(n.startswith("DEMO_") for n in os.environ))

### What just happened?

- With `override=False`, a variable that already exists is **kept**. That is what you want in CI and production: the platform sets real values, and a stray `.env` cannot replace them.
- **Consequence for notebooks:** if you edit `.env` while the kernel is running, `load_dotenv()` will not pick up the new value of a variable it already loaded. **Restart the kernel** (simplest), or reload with `override=True`.

### How the file is *found*

Called without a path, `load_dotenv()` searches for `.env` starting from a folder and walking **up** through the parents. `mastery.config.load_env()` always starts from the **current working directory** (`find_dotenv(usecwd=True)`). Jupyter starts the kernel in the notebook's folder (`modules/00-foundations/`), so the search walks up and finds `.env` at the repo root.

In [ ]:
print("Kernel working directory:", Path.cwd())
print("Search order for .env    :")
for folder in [Path.cwd(), *Path.cwd().parents][:4]:
    print("   ", folder / ".env", "← found" if (folder / ".env").is_file() else "")

---
## 5. Load the real `.env` with `mastery.config`

From now on, every notebook uses three helpers from [`src/mastery/config.py`](../../src/mastery/config.py):

- `load_env()` loads `.env` and returns **the path** it used (or `None`). It never returns values.
- `env_status(...)` describes each variable as *missing*, *placeholder*, or *set* with a masked preview.
- `require_env(...)` raises a friendly `MissingSettingError` if something is missing.

In [ ]:
from mastery.config import env_status, load_env

dotenv_path = load_env()
print("Loaded .env from:", dotenv_path or "no .env file found (see section 3)")

# Safe to print: values are masked (first 4 characters + length), never shown in full.
for name, status in env_status("GROQ_API_KEY", "LANGSMITH_API_KEY", "OPENAI_API_KEY").items():
    print(f"  {name:<18} {status}")

**You should now see** the path of your `.env` and `GROQ_API_KEY  set: gsk_… (NN chars)`. The other two are `missing`, which is correct for Module 00.

| You see | Meaning | Fix |
|---|---|---|
| `no .env file found` | `.env` is not in the repo root (or is named `.env.txt`) | Section 3 |
| `GROQ_API_KEY  missing` | The line is empty or misspelled | Open `.env`, check the name and that there is a value after `=`; save; **restart the kernel** |
| `placeholder (...)` | You pasted text such as `<your-api-key-here>` | Paste the real key |

Now the check every later notebook uses. It **does not crash** the notebook when the key is missing: it explains the fix and sets a flag that API cells check.

In [ ]:
from mastery.config import MissingSettingError, require_env

try:
    require_env("GROQ_API_KEY", dotenv_path=dotenv_path)
    HAVE_GROQ_KEY = True
    print("✅ GROQ_API_KEY is set. (Its value was not printed, and never will be.)")
except MissingSettingError as error:
    HAVE_GROQ_KEY = False
    print("⚠️ " + str(error))

### Why not just `print(os.environ["GROQ_API_KEY"])` to check?

Because outputs are **saved inside the `.ipynb` file** (Lesson 00.03). One accidental commit and the key is public. `mask_secret` shows enough to tell keys apart, and nothing more:

In [ ]:
from mastery.config import mask_secret

fake_key = "gsk_FAKE_" + "x" * 47          # A fake key with a realistic length.
print(mask_secret(fake_key))               # gsk_… (56 chars)
print(mask_secret(None))                   # <not set>

---
## 6. Which variables do the libraries read?

Libraries read their keys from the environment **by themselves**. You load `.env` once; you never pass keys around in code.

| Variable | Read by | Needed from | Source |
|---|---|---|---|
| `GROQ_API_KEY` | `langchain-groq` (`ChatGroq`), the `groq` SDK, `mastery.groq_catalog` | Module 00 | [ChatGroq](https://docs.langchain.com/oss/python/integrations/chat/groq) |
| `LANGSMITH_TRACING` | LangChain / LangSmith: `true` turns tracing on | Lesson 01.08 | [Tracing quickstart](https://docs.langchain.com/langsmith/observability-quickstart) |
| `LANGSMITH_API_KEY` | LangSmith client | Lesson 01.08 | same |
| `LANGSMITH_PROJECT` | LangSmith: which project receives traces | Lesson 01.08 | same |
| `LANGSMITH_ENDPOINT` | LangSmith: only for non-default regions or self-hosted | rarely | same |
| `OPENAI_API_KEY` | `langchain-openai` | optional (paid benchmark) | [paid benchmark guide](../../docs/paid-benchmark-openai-vs-gemini.md) |
| `GOOGLE_API_KEY` | `langchain-google-genai` | Module 11 | [paid benchmark guide](../../docs/paid-benchmark-openai-vs-gemini.md) |
| `MASTERY_MODEL_FAST`, `MASTERY_MODEL_REASONING`, … | the course (`mastery.config`) | Module 00 | [`.env.example`](../../.env.example) |

The `MASTERY_*` variables are **not secrets**. They keep model IDs out of the code, so switching models is a one-line `.env` change:

In [ ]:
from mastery.config import load_settings

settings = load_settings()        # Calls load_env() again; harmless, nothing is overridden.
print("Fast model     :", settings.model_fast)
print("Reasoning model:", settings.model_reasoning)
print("Tracing on?    :", settings.langsmith_tracing)

---
## 7. Fallback: type the key with `getpass`

Sometimes you can't use a `.env` file (a shared machine, a quick demo). `getpass.getpass()` asks for the value **without echoing it**, and it lives only in this kernel's memory until restart. This is the same pattern the [ChatGroq docs](https://docs.langchain.com/oss/python/integrations/chat/groq) show.

It is switched off by default, because an input box would block **Restart & Run All**. Set `USE_GETPASS = True`, run the cell, paste the key into the box, press **Enter**.

In [ ]:
import getpass

USE_GETPASS = False   # Set to True only when you want to type a key interactively.

if USE_GETPASS and not HAVE_GROQ_KEY:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your Groq API key (input is hidden): ")
    HAVE_GROQ_KEY = True
    print("Key stored in this kernel's memory only. It disappears when the kernel restarts.")
else:
    print("getpass fallback not used (USE_GETPASS =", USE_GETPASS, "| key already set =", HAVE_GROQ_KEY, ")")

> The IPython magic `%load_ext dotenv` + `%dotenv` also loads `.env` in one line ([python-dotenv docs](https://pypi.org/project/python-dotenv/)). The course uses `mastery.config.load_env()` instead, because it adds key checks and friendly errors.

---
## 8. If a key leaks: the playbook

A key "leaks" the moment it is in a commit, a public notebook output, a screenshot, or a chat. Act in this order:

1. **Rotate first.** In https://console.groq.com/keys, **delete** the leaked key and create a new one. Update `.env`. Deleting the commit is **not** enough: anyone may already have copied the key, and git history keeps old versions.
2. **Check usage** in the Groq console for activity you don't recognize.
3. **Find where it went.** In Git Bash, search history for a key prefix:
   ```bash
   git log -p -S 'gsk_' --oneline     # commits that added or removed text containing gsk_
   ```
4. **Clean up** the file (and, for a public repo, the history) only *after* rotating. Rewriting history is an advanced, destructive operation: read GitHub's guidance first and never do it on a shared branch without warning collaborators.

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| Edited `.env` but the old value is still used | `override=False` keeps already-loaded values | Restart the kernel |
| Key "missing" although it is in `.env` | Typo in the name, file saved as `.env.txt`, or the file is in another folder | `ls -a` in the repo root; compare the name with `.env.example` |
| Key is "set" but Groq says *Invalid API Key* (401) | Only part of the key was pasted, or the key was deleted in the console | Copy the key again (or create a new one); check the length shown by `env_status` |
| `.env` shows up in `git status` | It is not ignored | Run `git check-ignore -v .env`; never `git add -f .env` |
| A key appears in a notebook output | It was printed | Clear outputs, then follow the leak playbook |

## ⚠️ Production notes

> ⚠️ **Production note: `.env` files are a development shortcut.**
> A plain-text file of secrets on a laptop is fine for learning, not for servers. In production: inject secrets from a secret manager (your cloud's secret store, or your CI/CD platform's encrypted secrets) as environment variables, give each service its own key with the smallest permissions, rotate keys on a schedule, and never bake them into container images. Lesson 12.05 covers this for the course service. See [docs/production-notes.md](../../docs/production-notes.md).

> ⚠️ **Production note: `getpass` is for interactive sessions only.**
> A key typed into a notebook exists only in memory and can't be audited or rotated centrally. In production: the same secret-manager approach as above.

## 🏋️ Exercises

1. **(Easy)** Add `LANGSMITH_PROJECT=m00-foundations` to your `.env`, restart the kernel, and confirm that `load_settings().langsmith_project` changed.
2. **(Easy)** Run `git status` in Git Bash. Confirm that `.env` is **not** listed.
3. **(Medium)** Using `dotenv_values`, write `compare_env_files(a, b)` that returns the variable **names** present in `.env.example` but missing from `.env` (names only, never values).
4. **(Stretch)** Read `tests/unit/test_config.py`. Add a test that proves `env_status` reports a value of only spaces as `missing`. Run it with `python -m pytest tests/unit/test_config.py` in Git Bash.

<details>
<summary><b>Show solutions</b></summary>

**3.**
```python
def compare_env_files(example: Path, actual: Path) -> list[str]:
    """Names defined in `example` but absent (or empty) in `actual`. Never returns values."""
    expected = dotenv_values(example)
    present = {k for k, v in dotenv_values(actual).items() if v}
    return sorted(name for name in expected if name not in present)

print(compare_env_files(REPO_ROOT / ".env.example", REPO_ROOT / ".env"))
```

**4.**
```python
def test_env_status_whitespace_is_missing(clean_env):
    clean_env.setenv("GROQ_API_KEY", "   ")
    assert env_status("GROQ_API_KEY")["GROQ_API_KEY"] == "missing"
```
</details>

## ✅ Checkpoint

- [ ] `git check-ignore -v .env` prints a rule.
- [ ] Section 5 shows `GROQ_API_KEY  set: gsk_… (NN chars)` and the ✅ message.
- [ ] You can explain `override=False` and why you restart the kernel after editing `.env`.
- [ ] You know the four steps of the leak playbook, starting with **rotate**.

## 🔎 Docs drill (5 min)

Which environment variable does `ChatGroq` read for its key, and where does it say so? Check three places: the API reference, `ChatGroq.model_fields["groq_api_key"]`, and the installed source. Which of the three actually shows the variable name?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. API reference: search `ChatGroq` on reference.langchain.com; the `api_key` parameter says it's read from `GROQ_API_KEY` when not passed.
2. `ChatGroq.model_fields["groq_api_key"]` shows `alias='api_key'` and a `default_factory`, **not** the variable name: a factory is a function, so the name is hidden inside it.
3. Source: `inspect.getsourcefile(ChatGroq)`, then search `secret_from_env`: `secret_from_env("GROQ_API_KEY", default=None)`.
4. Answer: `GROQ_API_KEY`, visible in the reference and the source, not in `model_fields`. When a field has a `default_factory`, read the source.

</details>

## 🔎 Research checkpoint

Open the [LangSmith tracing quickstart](https://docs.langchain.com/langsmith/observability-quickstart). Which variable would you set if your LangSmith account were in the EU region? Note the answer in `docs/research-log.md`.

## Summary

- Config lives in the environment; `.env` is a local convenience, `.env.example` is the documentation.
- `load_dotenv` never overrides existing variables unless you ask. Restart the kernel after editing `.env`.
- Check keys with `require_env` / `env_status`; **never print a secret**.
- Leaked key? **Rotate first.**

**Next:** [00.05 · First contact: Groq + LangChain](00-05-first-groq-call.ipynb), where your new key makes its first call.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 00.05 · First contact: Groq + LangChain** → [`00-05-first-groq-call.ipynb`](00-05-first-groq-call.ipynb).

[Module 00 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)